# v3 continual run (GPU 0)
Hindi (loaded) → Yoruba → Chinese → Hindi → ... for 24 phases (8 cycles).
Each phase: fresh AdamW, warmup → stable at 2e-4 until validation plateaus → 2,000-step cooldown to 2e-5 → switch.
Script-masked loss and decoding, weight decay 0.1 on weight matrices, divergence guard.
Checkpoints: `runs/v3_continual_s0/ckpt/pXXX_<lang>.pt`, one per phase. Rerunning resumes after the last saved phase.

In [1]:
# Run this FIRST in a fresh kernel: it pins the notebook to GPU 0 before torch starts.
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO = "/workspace/shreshth"
os.chdir(REPO); sys.path.insert(0, REPO)
from common import *
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")

device: cuda | NVIDIA GeForce RTX 3090


In [5]:
pwd

'/workspace/shreshth'

In [8]:
ls

 Untitled-1.ipynb      data/                         tokenizer_train.ipynb
 __pycache__/          encoder_decoder_hindi.ipynb   train_continual_gpu0.ipynb
'analysis (3).ipynb'   runs/                         train_controls_gpu1.ipynb
 common.py             tokenizer/


In [12]:
SMOKE = False      # True first, for a quick end-to-end test
C = RunConfig(run_name="v3_continual_s0", mode="continual", n_phases=24)

if SMOKE:   # tiny phases, ~10 minutes, separate run folder
    C.run_name += "_smoke"; C.n_phases = 3; C.eval_every = 50; C.warmup_steps = 50
    C.min_steps = 100; C.cooldown_steps = 100; C.max_stable_steps = 300

print(C)

RunConfig(run_name='v3_continual_s0', mode='continual', order=('hi', 'yo', 'zh'), init_ckpt='runs/en-hi_s0/best.pt', n_phases=24, seed=0, max_tokens=16000, lr=0.0002, min_lr=2e-05, warmup_steps=500, cooldown_steps=2000, min_steps=2000, max_stable_steps=20000, eval_every=500, plateau_window=4, plateau_tol=0.005, weight_decay=0.1, grad_clip=1.0, masked=True, diverge_factor=1.15, max_retries=2, time_cap_h=30.0)


In [10]:
D = load_data(train_langs=["hi", "yo", "zh"])

  hi: 993,016 train / 2,000 val
  zh: 755,626 train / 2,000 val
  yo: 227,670 train / 2,000 val
data loaded in 36s


In [ ]:
model = train_run(C, D)

 RUN v3_continual_s0 | mode continual | 24 phases after phase 0 | device cuda
 phase order: hi (loaded) → yo → zh → hi → yo → zh → hi → yo → ...
 each phase: fresh AdamW (wd 0.1 on matrices), warmup 500 → lr 0.0002 → plateau (<0.5% over 4-eval windows, after 2,000, cap 20,000) → cooldown 2,000 steps to 2e-05
 masked (script-restricted) loss & decoding: True | divergence guard: >1.15x best twice → retry at half lr (max 2)
phase 0: evaluating the loaded Hindi model on all languages...

 PHASE 0 DONE | trained: hi | loaded model, no training
--------------------------------------------------------------------------------------
   lang      val       Δ |   chrF      Δ |  chrF (unrestricted)  val (unrestr.)
   hi      1.500         |   45.6        |                 45.6           1.557  ◀ trained
   zh      9.050         |    0.3        |                  0.9          10.565
   yo      8.331         |    1.1        |                  0.6           8.852

▶ PHASE 1/24 | training yo (227,670 